# 05 Holt-Winters Seasonality

Holt-Winters exponential smoothing adds seasonal factors to Holt's level and trend states.

Use additive Holt-Winters when seasonal swings are roughly constant:

$$\hat y_{T+\tau|T} = l_T + \tau b_T + s_{season}.$$

Use multiplicative Holt-Winters when seasonal swings grow with the level:

$$\hat y_{T+\tau|T} = (l_T + \tau b_T)s_{season}.$$

In this course, the most important skill is recognizing which seasonal structure is plausible and using software or code to estimate the constants reliably.

## Holt-Winters State Updates

Let `L` be the seasonal period, such as `L = 4` for quarterly data. Holt-Winters keeps three states: the level `l_t`, trend `b_t`, and seasonal factor `s_t`.

For additive seasonality, the recursive updates are

$$l_t = \alpha(y_t - s_{t-L}) + (1-\alpha)(l_{t-1}+b_{t-1}),$$

$$b_t = \gamma(l_t-l_{t-1}) + (1-\gamma)b_{t-1},$$

$$s_t = \delta(y_t-l_t) + (1-\delta)s_{t-L}.$$

The forecast is

$$\hat y_{T+\tau|T} = l_T + \tau b_T + s_{T+\tau-Lk},$$

where `k` is chosen so that the seasonal index refers to the matching future season.

For multiplicative seasonality, the same idea is used with ratios instead of differences:

$$l_t = \alpha\frac{y_t}{s_{t-L}} + (1-\alpha)(l_{t-1}+b_{t-1}),$$

$$s_t = \delta\frac{y_t}{l_t} + (1-\delta)s_{t-L},$$

and

$$\hat y_{T+\tau|T} = (l_T + \tau b_T)s_{T+\tau-Lk}.$$

The lecture emphasizes additive calculations in detail and uses multiplicative Holt-Winters mainly to explain cases where seasonal swings grow with the level.


## How the Initial Seasonal Factors Are Built

Before the recursive Holt-Winters updates begin, software needs starting values for the level, trend, and seasonal factors. A practical initialization used in the lecture is:

1. Fit a straight line to the available series to estimate an initial level and trend.
2. Remove that trend from each observation.
3. Average the detrended values by season, then normalize the seasonal factors.

For additive seasonality, detrending means `observed - trend line`, and the seasonal factors are normalized to average zero. For multiplicative seasonality, detrending means `observed / trend line`, and the seasonal factors are normalized to average one.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from checks import check_between, check_close, check_columns
from smoothing_utils import (
    accuracy_measures, initial_level_mean, initial_line,
    simple_es, optimize_simple_es,
    holt_trend, optimize_holt, holt_forecast_table,
    holt_winters, optimize_holt_winters, holt_winters_forecast,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 4)
DATA_DIR = Path('data')

## Additive Seasonality: Quarterly Bike Sales

The bike series has a repeating quarterly pattern where the high and low quarters differ by roughly similar amounts across years. That is the additive case.

In [ ]:
bike = pd.read_csv(DATA_DIR / 'bike_sales.csv')
y_bike = bike['BikeSales'].to_numpy()
alpha, gamma, delta, fit_add, metrics_add = optimize_holt_winters(y_bike, period=4, kind='additive')
print(f'Additive HW constants: alpha={alpha:.3f}, gamma={gamma:.3f}, delta={delta:.3f}')
pd.Series(metrics_add).to_frame('value')

In [ ]:
season_index = (np.arange(len(y_bike)) % 4) + 1
l0_init, b0_init = initial_line(y_bike, n=len(y_bike))
trend_line = l0_init + b0_init * np.arange(1, len(y_bike) + 1)

initial_additive = (
    pd.DataFrame({'quarter': season_index, 'detrended_value': y_bike - trend_line})
    .groupby('quarter')['detrended_value']
    .mean()
)
initial_additive = initial_additive - initial_additive.mean()

pd.DataFrame({
    'quarter': initial_additive.index,
    'initial_additive_seasonal_factor': initial_additive.values,
}).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(bike['Time'], y_bike, marker='o', label='Observed')
ax.plot(bike['Time'], fit_add['yhat_one_step'], marker='o', label='Additive HW one-step forecast')
ax.set_xlabel('Quarter')
ax.set_ylabel('Bike sales')
ax.legend()
plt.tight_layout()

In [ ]:
holt_winters_forecast(fit_add, h=4).round(2)

## Multiplicative Seasonality: Sports Drink Sales

The sports drink series has larger seasonal swings when the overall level is higher. That is the multiplicative case. We can fit both additive and multiplicative versions and compare accuracy, but the plot already gives a strong modeling clue.

In [ ]:
drink = pd.read_csv(DATA_DIR / 'sports_drink_sales.csv')
y_drink = drink['DrinkSales'].to_numpy()

alpha_a, gamma_a, delta_a, fit_drink_add, metrics_drink_add = optimize_holt_winters(y_drink, period=4, kind='additive')
alpha_m, gamma_m, delta_m, fit_drink_mult, metrics_drink_mult = optimize_holt_winters(y_drink, period=4, kind='multiplicative')

pd.DataFrame({'additive': metrics_drink_add, 'multiplicative': metrics_drink_mult})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(drink['Time'], y_drink, marker='o', label='Observed')
ax.plot(drink['Time'], fit_drink_add['yhat_one_step'], label='Additive HW')
ax.plot(drink['Time'], fit_drink_mult['yhat_one_step'], label='Multiplicative HW')
ax.set_xlabel('Quarter')
ax.set_ylabel('Sports drink sales')
ax.legend()
plt.tight_layout()

In [ ]:
holt_winters_forecast(fit_drink_mult, h=4).round(2)

## Model Choice Summary

- No trend and no seasonality: start with simple exponential smoothing.
- Trend but no seasonality: start with Holt's method.
- Trend and constant seasonal swings: start with additive Holt-Winters.
- Trend and growing seasonal swings: start with multiplicative Holt-Winters.

After choosing a plausible family, use accuracy measures and residual plots to check whether the model is doing a reasonable job.